In [2]:

from pathlib import Path
import pandas as pd

# Project directories
PROJECT_ROOT = Path.cwd()

# If the notebook is opened from the notebooks folder,
# the project root is its parent directory.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

ANNOTATIONS_DIR = PROJECT_ROOT / "data" / "annotations"

# Load the training metadata
annotations = pd.read_csv(
    ANNOTATIONS_DIR / "annotations_train.csv"
)

labels = pd.read_csv(
    ANNOTATIONS_DIR / "image_labels_train.csv"
)

# Inspect dataset dimensions
print("BOUNDING-BOX ANNOTATIONS")
print("Shape:", annotations.shape)
print("Columns:", annotations.columns.tolist())

print("\nCLASSIFICATION LABELS")
print("Shape:", labels.shape)
print("Columns:", labels.columns.tolist())


BOUNDING-BOX ANNOTATIONS
Shape: (69052, 7)
Columns: ['image_id', 'rad_id', 'class_name', 'x_min', 'y_min', 'x_max', 'y_max']

CLASSIFICATION LABELS
Shape: (45000, 30)
Columns: ['image_id', 'rad_id', 'Aortic enlargement', 'Atelectasis', 'Calcification', 'Cardiomegaly', 'Clavicle fracture', 'Consolidation', 'Edema', 'Emphysema', 'Enlarged PA', 'ILD', 'Infiltration', 'Lung Opacity', 'Lung cavity', 'Lung cyst', 'Mediastinal shift', 'Nodule/Mass', 'Pleural effusion', 'Pleural thickening', 'Pneumothorax', 'Pulmonary fibrosis', 'Rib fracture', 'Other lesion', 'COPD', 'Lung tumor', 'Pneumonia', 'Tuberculosis', 'Other diseases', 'No finding']


In [3]:

# Basic dataset statistics
print("Unique training images in annotations:",
      annotations["image_id"].nunique())

print("Unique training images in classification labels:",
      labels["image_id"].nunique())

# Identify rows containing actual bounding boxes
bbox_columns = ["x_min", "y_min", "x_max", "y_max"]

has_bbox = annotations[bbox_columns].notna().all(axis=1)

print("Rows with bounding boxes:", has_bbox.sum())

print("Unique images with bounding boxes:",
      annotations.loc[has_bbox, "image_id"].nunique())

# Count findings by class
print("\nMost frequent annotated findings:")
print(
    annotations.loc[has_bbox, "class_name"]
    .value_counts()
    .head(15)
)


Unique training images in annotations: 15000
Unique training images in classification labels: 15000
Rows with bounding boxes: 37367
Unique images with bounding boxes: 4522

Most frequent annotated findings:
class_name
Aortic enlargement    7193
Cardiomegaly          5443
Pleural thickening    4884
Pulmonary fibrosis    4659
Nodule/Mass           2611
Lung Opacity          2493
Pleural effusion      2483
Other lesion          2228
Infiltration          1247
ILD                   1015
Calcification          967
Consolidation          556
Rib fracture           300
Atelectasis            280
Mediastinal shift      270
Name: count, dtype: int64


In [4]:
# The notebook is inside notebooks/, so go up one level
PROJECT_ROOT = Path.cwd().parent

ANNOTATIONS_PATH = (
    PROJECT_ROOT / "data" / "annotations"
    / "annotations_train.csv"
)

LABELS_PATH = (
    PROJECT_ROOT / "data" / "annotations"
    / "image_labels_train.csv"
)

# Check paths before loading anything
print("Project root:", PROJECT_ROOT)
print("Annotations CSV exists:", ANNOTATIONS_PATH.exists())
print("Labels CSV exists:", LABELS_PATH.exists())

if not ANNOTATIONS_PATH.exists():
    raise FileNotFoundError(
        f"Cannot find annotations CSV: {ANNOTATIONS_PATH}"
    )

if not LABELS_PATH.exists():
    raise FileNotFoundError(
        f"Cannot find labels CSV: {LABELS_PATH}"
    )

annotations = pd.read_csv(ANNOTATIONS_PATH)
labels = pd.read_csv(LABELS_PATH)

print("\nBoth CSV files loaded successfully!")
print("Annotations shape:", annotations.shape)
print("Labels shape:", labels.shape)


Project root: c:\Users\johnm\chest-xray-ai-annotation
Annotations CSV exists: True
Labels CSV exists: True

Both CSV files loaded successfully!
Annotations shape: (69052, 7)
Labels shape: (45000, 30)


In [5]:

# Keep only rows with complete bounding-box coordinates
valid_boxes = annotations.dropna(
    subset=["x_min", "y_min", "x_max", "y_max"]
).copy()

# Get unique images that have bounding-box annotations
annotated_image_ids = set(
    valid_boxes["image_id"].unique()
)

# Get images present in the classification labels CSV
label_image_ids = set(
    labels["image_id"].dropna().unique()
)

# Keep images represented in both CSV files
eligible_ids = sorted(
    annotated_image_ids & label_image_ids
)

print("Unique images with valid bounding boxes:",
      len(annotated_image_ids))

print("Images also present in classification labels:",
      len(eligible_ids))

print("\nSample eligible image IDs:")
print(eligible_ids[:10])


Unique images with valid bounding boxes: 4522
Images also present in classification labels: 4522

Sample eligible image IDs:
['0005e8e3701dfb1dd93d53e2ff537b6e', '0007d316f756b3fa0baea2ff514ce945', '000d68e42b71d3eac10ccc077aba07c1', '00150343289f317a0ad5629d5b7d9ef9', '001d127bad87592efe45a5c7678f8b8d', '0021df30f3fddef551eb3df4354b1d06', '0046f681f078851293c4e710c4466058', '0061cf6d35e253b6e7f03940592cc35e', '00675cd546313f912cadd4ad54415d69', '008b3176a7248a0a189b5731ac8d2e95']


In [6]:

# Select 300 unique images reproducibly
import random

random.seed(42)

selected_image_ids = random.sample(eligible_ids, 300)

output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

manifest = pd.DataFrame({
    "image_id": selected_image_ids
})

manifest_path = output_dir / "subset_manifest.csv"
manifest.to_csv(manifest_path, index=False)

print("Selected unique images:", manifest["image_id"].nunique())
print("Manifest saved to:", manifest_path)
print("\nFirst 10 selected image IDs:")
print(manifest.head(10).to_string(index=False))


Selected unique images: 300
Manifest saved to: c:\Users\johnm\chest-xray-ai-annotation\data\processed\subset_manifest.csv

First 10 selected image IDs:
                        image_id
32e735ff23d1c5bf5d2d1aa51937ca82
0b1b897b1e1e170f1b5fd7aeff553afa
80234017eb6caeada9e14d421eb0b8f8
723f93a3f0e3905c498839d23b961631
6833c509c10b569d6167db70b83ee5d5
3f0efef8e4a56b73d2b82e83e50c0b90
2fd3d0e3b5de5535994ac58f2bd175da
fc47195078badf33f2575c3f0db648ab
285cdfec4d725d327266833adf3ba9d5
c32573547f924b6b663798730d915724


In [7]:
manifest = pd.read_csv(manifest_path)

print("Total rows:", len(manifest))
print("Unique image IDs:", manifest["image_id"].nunique())
print("Missing image IDs:", manifest["image_id"].isna().sum())

Total rows: 300
Unique image IDs: 300
Missing image IDs: 0


In [8]:

from pathlib import Path
import pandas as pd

# Locate project root
ROOT = Path.cwd()
while not (ROOT / "data" / "annotations").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Project root not found.")
    ROOT = ROOT.parent

image_dir = ROOT / "data" / "images" / "train"
annotation_dir = ROOT / "data" / "annotations"
processed_dir = ROOT / "data" / "processed"

# Find DICOMs in train and all subfolders
dicom_files = sorted(image_dir.rglob("*.dicom"))

if not dicom_files:
    raise FileNotFoundError(f"No DICOM files found under {image_dir}")

# Map image IDs to their actual file paths
image_paths = {}
duplicates = []

for p in dicom_files:
    if p.stem in image_paths:
        duplicates.append(p.stem)
    else:
        image_paths[p.stem] = p

image_ids = sorted(image_paths)

# Load annotations and labels
labels = pd.read_csv(annotation_dir / "image_labels_train.csv")
annotations = pd.read_csv(annotation_dir / "annotations_train.csv")

label_ids = set(labels["image_id"].dropna().astype(str))

box_cols = ["x_min", "y_min", "x_max", "y_max"]
valid_boxes = annotations.dropna(subset=box_cols)
bbox_ids = set(valid_boxes["image_id"].dropna().astype(str))

# Build manifest from the images actually downloaded
manifest = pd.DataFrame({"image_id": image_ids})
manifest["dicom_path"] = manifest["image_id"].map(
    lambda i: image_paths[i].relative_to(ROOT).as_posix()
)
manifest["has_classification_label"] = manifest["image_id"].isin(label_ids)
manifest["has_bbox"] = manifest["image_id"].isin(bbox_ids)

processed_dir.mkdir(parents=True, exist_ok=True)
out = processed_dir / "downloaded_manifest.csv"
manifest.to_csv(out, index=False)

print("DICOM files found:", len(dicom_files))
print("Unique image IDs:", len(image_ids))
print("Duplicate IDs across folders:", len(duplicates))
print("Images with classification labels:",
      int(manifest["has_classification_label"].sum()))
print("Images with bounding boxes:",
      int(manifest["has_bbox"].sum()))
print("Images without bounding boxes:",
      int((~manifest["has_bbox"]).sum()))
print("Manifest saved to:", out)


DICOM files found: 301
Unique image IDs: 297
Duplicate IDs across folders: 4
Images with classification labels: 297
Images with bounding boxes: 100
Images without bounding boxes: 197
Manifest saved to: c:\Users\johnm\chest-xray-ai-annotation\data\processed\downloaded_manifest.csv


In [9]:

from collections import defaultdict

ROOT = Path.cwd()
while not (ROOT / "data" / "images").exists():
    ROOT = ROOT.parent

image_dir = ROOT / "data" / "images" / "train"
groups = defaultdict(list)

for p in image_dir.rglob("*.dicom"):
    groups[p.stem].append(p)

for image_id, paths in groups.items():
    if len(paths) > 1:
        print(f"\n{image_id}")
        for p in paths:
            print(" ", p.relative_to(ROOT))



1b6b4e0d7f5f7d7ad2bf8b70783244be
  data\images\train\vindr_subset\1b6b4e0d7f5f7d7ad2bf8b70783244be.dicom
  data\images\train\vindr_subset(1)\1b6b4e0d7f5f7d7ad2bf8b70783244be.dicom

886159a41e7589ba0d0713980591e27d
  data\images\train\vindr_subset\886159a41e7589ba0d0713980591e27d.dicom
  data\images\train\vindr_subset(1)\886159a41e7589ba0d0713980591e27d.dicom

d1840dda98dafc76f201fa30bda3ef94
  data\images\train\vindr_subset\d1840dda98dafc76f201fa30bda3ef94.dicom
  data\images\train\vindr_subset(1)\d1840dda98dafc76f201fa30bda3ef94.dicom

e33f67bb274e75146faa95baaa6fbd47
  data\images\train\vindr_subset\e33f67bb274e75146faa95baaa6fbd47.dicom
  data\images\train\vindr_subset(1)\e33f67bb274e75146faa95baaa6fbd47.dicom


In [10]:

from pathlib import Path
import pandas as pd

# Find project root
ROOT = Path.cwd()
while not (ROOT / "data" / "annotations").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Project root not found")
    ROOT = ROOT.parent

image_dir = ROOT / "data" / "images" / "train"
ann_dir = ROOT / "data" / "annotations"
processed_dir = ROOT / "data" / "processed"

# Inspect all DICOMs in all three sets
dicom_files = sorted(image_dir.rglob("*.dicom"))

file_df = pd.DataFrame([
    {
        "image_id": p.stem,
        "path": p,
        "folder": p.parent.name,
        "size_bytes": p.stat().st_size
    }
    for p in dicom_files
])

if file_df.empty:
    raise FileNotFoundError(f"No DICOM files found in {image_dir}")

print("Total DICOM files:", len(file_df))
print("Unique image IDs:", file_df["image_id"].nunique())
print("\nFiles by folder:")
print(file_df["folder"].value_counts().to_string())

duplicates = file_df[
    file_df.duplicated("image_id", keep=False)
].sort_values("image_id")

print("\nDuplicate image IDs:", duplicates["image_id"].nunique())
if not duplicates.empty:
    print(duplicates[["image_id", "folder", "size_bytes"]].to_string(index=False))

# Load annotations
labels = pd.read_csv(ann_dir / "image_labels_train.csv")
annotations = pd.read_csv(ann_dir / "annotations_train.csv")

label_ids = set(labels["image_id"].dropna().astype(str))
valid_boxes = annotations.dropna(
    subset=["x_min", "y_min", "x_max", "y_max"]
)
bbox_ids = set(valid_boxes["image_id"].dropna().astype(str))

# One manifest row per unique image ID.
# Prefer the first available copy; preserve all files on disk.
unique_files = file_df.drop_duplicates("image_id", keep="first").copy()
unique_files["dicom_path"] = unique_files["path"].map(
    lambda p: p.relative_to(ROOT).as_posix()
)
unique_files["has_classification_label"] = (
    unique_files["image_id"].isin(label_ids)
)
unique_files["has_bbox"] = unique_files["image_id"].isin(bbox_ids)

processed_dir.mkdir(parents=True, exist_ok=True)
out = processed_dir / "downloaded_manifest.csv"
unique_files[
    ["image_id", "dicom_path", "has_classification_label", "has_bbox"]
].to_csv(out, index=False)

print("\n--- Annotation coverage ---")
print("Unique downloaded images:", len(unique_files))
print("With classification labels:",
      int(unique_files["has_classification_label"].sum()))
print("With bounding boxes:",
      int(unique_files["has_bbox"].sum()))
print("Without bounding boxes:",
      int((~unique_files["has_bbox"]).sum()))
print("Manifest saved:", out)


Total DICOM files: 301
Unique image IDs: 297

Files by folder:
folder
vindr_subset(1)    101
vindr_subset       100
vindr_subset2      100

Duplicate image IDs: 4
                        image_id          folder  size_bytes
1b6b4e0d7f5f7d7ad2bf8b70783244be    vindr_subset    19160580
1b6b4e0d7f5f7d7ad2bf8b70783244be vindr_subset(1)    19160580
886159a41e7589ba0d0713980591e27d    vindr_subset     7355034
886159a41e7589ba0d0713980591e27d vindr_subset(1)      212447
d1840dda98dafc76f201fa30bda3ef94    vindr_subset    16899180
d1840dda98dafc76f201fa30bda3ef94 vindr_subset(1)     1244637
e33f67bb274e75146faa95baaa6fbd47    vindr_subset     9691322
e33f67bb274e75146faa95baaa6fbd47 vindr_subset(1)      409055

--- Annotation coverage ---
Unique downloaded images: 297
With classification labels: 297
With bounding boxes: 100
Without bounding boxes: 197
Manifest saved: c:\Users\johnm\chest-xray-ai-annotation\data\processed\downloaded_manifest.csv


In [11]:

from pathlib import Path
import pydicom

ROOT = Path.cwd()
while not (ROOT / "data" / "images").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Project root not found")
    ROOT = ROOT.parent

image_dir = ROOT / "data" / "images" / "train"

duplicate_ids = [
    "1b6b4e0d7f5f7d7ad2bf8b70783244be",
    "886159a41e7589ba0d0713980591e27d",
    "d1840dda98dafc76f201fa30bda3ef94",
    "e33f67bb274e75146faa95baaa6fbd47",
]

for image_id in duplicate_ids:
    print(f"\nImage: {image_id}")

    for path in sorted(image_dir.rglob(f"{image_id}.dicom")):
        size_mb = path.stat().st_size / (1024 * 1024)

        try:
            ds = pydicom.dcmread(str(path), stop_before_pixels=True)
            status = "DICOM header readable"
            if "Rows" in ds and "Columns" in ds:
                status += f" | {ds.Rows}x{ds.Columns}"
        except Exception as e:
            status = f"READ ERROR: {e}"

        print(f"  {path.parent.name}: {size_mb:.2f} MB | {status}")



Image: 1b6b4e0d7f5f7d7ad2bf8b70783244be
  vindr_subset: 18.27 MB | DICOM header readable | 3141x3050
  vindr_subset(1): 18.27 MB | DICOM header readable | 3141x3050

Image: 886159a41e7589ba0d0713980591e27d
  vindr_subset: 7.01 MB | DICOM header readable | 2372x2504
  vindr_subset(1): 0.20 MB | DICOM header readable | 2372x2504

Image: d1840dda98dafc76f201fa30bda3ef94
  vindr_subset: 16.12 MB | DICOM header readable | 3095x2730
  vindr_subset(1): 1.19 MB | DICOM header readable | 3095x2730

Image: e33f67bb274e75146faa95baaa6fbd47
  vindr_subset: 9.24 MB | DICOM header readable | 2430x1994
  vindr_subset(1): 0.39 MB | DICOM header readable | 2430x1994


In [12]:

from collections import defaultdict

# Find project root
ROOT = Path.cwd()
while not (ROOT / "data" / "annotations").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Project root not found")
    ROOT = ROOT.parent

image_dir = ROOT / "data" / "images" / "train"
ann_dir = ROOT / "data" / "annotations"
processed_dir = ROOT / "data" / "processed"

# Group all downloaded DICOMs by image ID
groups = defaultdict(list)
for path in image_dir.rglob("*.dicom"):
    groups[path.stem].append(path)

# Load annotation sources
labels = pd.read_csv(ann_dir / "image_labels_train.csv")
annotations = pd.read_csv(ann_dir / "annotations_train.csv")

label_ids = set(labels["image_id"].dropna().astype(str))
box_cols = ["x_min", "y_min", "x_max", "y_max"]
valid_boxes = annotations.dropna(subset=box_cols)
bbox_ids = set(valid_boxes["image_id"].dropna().astype(str))

chosen = []
issues = []

for image_id, paths in sorted(groups.items()):
    # Prefer larger files; try alternatives if pixel decoding fails
    candidates = sorted(paths, key=lambda p: p.stat().st_size, reverse=True)
    selected = None

    for path in candidates:
        if len(paths) > 1:
            try:
                ds = pydicom.dcmread(str(path))
                _ = ds.pixel_array  # Test actual pixel decoding
                selected = path
                break
            except Exception as e:
                issues.append((image_id, str(path), str(e)))
        else:
            selected = path
            break

    if selected is None:
        print(f"WARNING: No decodable duplicate copy: {image_id}")
        continue

    chosen.append({
        "image_id": image_id,
        "dicom_path": selected.relative_to(ROOT).as_posix(),
        "file_size_bytes": selected.stat().st_size,
        "has_classification_label": image_id in label_ids,
        "has_bbox": image_id in bbox_ids,
    })

manifest = pd.DataFrame(chosen)
processed_dir.mkdir(parents=True, exist_ok=True)
out = processed_dir / "downloaded_manifest.csv"
manifest.to_csv(out, index=False)

print("Total files found:", sum(map(len, groups.values())))
print("Unique image IDs found:", len(groups))
print("Images in final manifest:", len(manifest))
print("Images with classification labels:",
      int(manifest["has_classification_label"].sum()))
print("Images with bounding boxes:",
      int(manifest["has_bbox"].sum()))
print("Duplicate candidates that failed decoding:", len(issues))
print("Manifest saved:", out)

if issues:
    print("\nDecode failures:")
    for image_id, path, error in issues:
        print(image_id, "|", path, "|", error[:180])


Total files found: 301
Unique image IDs found: 297
Images in final manifest: 297
Images with classification labels: 297
Images with bounding boxes: 100
Duplicate candidates that failed decoding: 0
Manifest saved: c:\Users\johnm\chest-xray-ai-annotation\data\processed\downloaded_manifest.csv


In [13]:

from pathlib import Path
import pydicom

ROOT = Path.cwd()
while not (ROOT / "data" / "images").exists():
    ROOT = ROOT.parent

image_id = "886159a41e7589ba0d0713980591e27d"
path = (
    ROOT / "data" / "images" / "train" /
    "vindr_subset" / f"{image_id}.dicom"
)

ds = pydicom.dcmread(str(path))
pixels = ds.pixel_array

print("Pixel decoding successful")
print("Image shape:", pixels.shape)
print("Pixel dtype:", pixels.dtype)


Pixel decoding successful
Image shape: (2372, 2504)
Pixel dtype: uint16


In [14]:

from pathlib import Path
from collections import defaultdict
import pandas as pd
import pydicom

ROOT = Path.cwd()
while not (ROOT / "data" / "annotations").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Project root not found")
    ROOT = ROOT.parent

image_dir = ROOT / "data" / "images" / "train"
ann_dir = ROOT / "data" / "annotations"
processed_dir = ROOT / "data" / "processed"

groups = defaultdict(list)
for p in image_dir.rglob("*.dicom"):
    groups[p.stem].append(p)

labels = pd.read_csv(ann_dir / "image_labels_train.csv")
annotations = pd.read_csv(ann_dir / "annotations_train.csv")

label_ids = set(labels["image_id"].dropna().astype(str))
valid_boxes = annotations.dropna(
    subset=["x_min", "y_min", "x_max", "y_max"]
)
bbox_ids = set(valid_boxes["image_id"].dropna().astype(str))

rows = []
failures = []

for image_id, paths in sorted(groups.items()):
    # Try largest file first; fall back to another copy if needed
    candidates = sorted(paths, key=lambda p: p.stat().st_size, reverse=True)
    selected = None
    error_messages = []

    for path in candidates:
        try:
            ds = pydicom.dcmread(str(path))
            _ = ds.pixel_array
            selected = path
            break
        except Exception as e:
            error_messages.append(f"{path.name}: {e}")

    if selected is None:
        failures.append((image_id, error_messages))
        continue

    rows.append({
        "image_id": image_id,
        "dicom_path": selected.relative_to(ROOT).as_posix(),
        "file_size_bytes": selected.stat().st_size,
        "has_classification_label": image_id in label_ids,
        "has_bbox": image_id in bbox_ids,
    })

manifest = pd.DataFrame(rows)
processed_dir.mkdir(parents=True, exist_ok=True)
out = processed_dir / "downloaded_manifest.csv"
manifest.to_csv(out, index=False)

print("DICOM files found:", sum(map(len, groups.values())))
print("Unique IDs found:", len(groups))
print("Images with decodable pixel data:", len(manifest))
print("Images with classification labels:",
      int(manifest["has_classification_label"].sum()))
print("Images with bounding boxes:",
      int(manifest["has_bbox"].sum()))
print("Images that failed decoding:", len(failures))
print("Manifest saved:", out)

if failures:
    print("\nFailed image IDs:")
    for image_id, _ in failures:
        print(image_id)


DICOM files found: 301
Unique IDs found: 297
Images with decodable pixel data: 295
Images with classification labels: 295
Images with bounding boxes: 100
Images that failed decoding: 2
Manifest saved: c:\Users\johnm\chest-xray-ai-annotation\data\processed\downloaded_manifest.csv

Failed image IDs:
250e4a4bfa51c58093c8a111a6adff14
49348e03337fc04b3ef5c4472c7c1df1


In [15]:

from pathlib import Path
import pydicom

ROOT = Path.cwd()
while not (ROOT / "data" / "images").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Project root not found")
    ROOT = ROOT.parent

image_dir = ROOT / "data" / "images" / "train"

failed_ids = [
    "250e4a4bfa51c58093c8a111a6adff14",
    "49348e03337fc04b3ef5c4472c7c1df1",
]

for image_id in failed_ids:
    print(f"\n--- {image_id} ---")
    matches = list(image_dir.rglob(f"{image_id}.dicom"))

    if not matches:
        print("File not found")
        continue

    for path in matches:
        print("Path:", path.relative_to(ROOT))
        print("Size (bytes):", path.stat().st_size)

        try:
            ds = pydicom.dcmread(str(path))
            print("Transfer syntax:",
                  getattr(ds.file_meta, "TransferSyntaxUID", "Unknown"))
            print("Has pixel data:", "PixelData" in ds)
            pixels = ds.pixel_array
            print("Decoded:", pixels.shape, pixels.dtype)
        except Exception as e:
            print("ERROR:", repr(e))



--- 250e4a4bfa51c58093c8a111a6adff14 ---
Path: data\images\train\vindr_subset(1)\250e4a4bfa51c58093c8a111a6adff14.dicom
Size (bytes): 11435054
Transfer syntax: 1.2.840.10008.1.2
Has pixel data: True
ERROR: ValueError('The number of bytes of pixel data is less than expected (11434574 vs 14256198 bytes) - the dataset may be corrupted, have an invalid group 0028 element value, or the transfer syntax may be incorrect')

--- 49348e03337fc04b3ef5c4472c7c1df1 ---
Path: data\images\train\vindr_subset(1)\49348e03337fc04b3ef5c4472c7c1df1.dicom
Size (bytes): 720350
Transfer syntax: 1.2.840.10008.1.2
Has pixel data: True
ERROR: ValueError('The number of bytes of pixel data is less than expected (719870 vs 13249792 bytes) - the dataset may be corrupted, have an invalid group 0028 element value, or the transfer syntax may be incorrect')


In [16]:

from pathlib import Path
import pandas as pd
import pydicom

ROOT = Path.cwd()
while not (ROOT / "data" / "processed").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Project root not found")
    ROOT = ROOT.parent

manifest_path = ROOT / "data" / "processed" / "downloaded_manifest.csv"
manifest = pd.read_csv(manifest_path)

valid_rows = []
failed_rows = []

for _, row in manifest.iterrows():
    path = ROOT / row["dicom_path"]
    try:
        ds = pydicom.dcmread(str(path))
        _ = ds.pixel_array
        valid_rows.append(row)
    except Exception as e:
        failed_rows.append((row["image_id"], str(e)))

valid_manifest = pd.DataFrame(valid_rows)
out = ROOT / "data" / "processed" / "validated_manifest.csv"
valid_manifest.to_csv(out, index=False)

print("Validated images:", len(valid_manifest))
print("Failed images:", len(failed_rows))
print("Images with bounding boxes:",
      int(valid_manifest["has_bbox"].sum()))
print("Saved:", out)

if failed_rows:
    print("Failed IDs:", [image_id for image_id, _ in failed_rows])


FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\johnm\\chest-xray-ai-annotation\\notebooks\\data\\processed\\downloaded_manifest.csv'

In [ ]:

from pathlib import Path
import pandas as pd
import pydicom

ROOT = Path.cwd()
while not (ROOT / "data" / "annotations").exists():
    ROOT = ROOT.parent

image_id = "113274a4ca3c2e72b7ff1bd3ce5006b6"

annotations = pd.read_csv(
    ROOT / "data" / "annotations" / "annotations_train.csv",
    dtype={"image_id": str},
)

image_dir = ROOT / "data" / "images" / "train"
manifest = pd.read_csv(
    ROOT / "data" / "processed" / "validated_manifest.csv",
    dtype={"image_id": str},
)

row = manifest.loc[manifest["image_id"] == image_id]

if row.empty:
    print("Image ID not found in validated manifest.")
else:
    dicom_path = ROOT / row.iloc[0]["dicom_path"]
    ds = pydicom.dcmread(str(dicom_path))

    boxes = annotations[
        annotations["image_id"] == image_id
    ].dropna(subset=["x_min", "y_min", "x_max", "y_max"])

    print("DICOM dimensions:", ds.Rows, "x", ds.Columns)
    print("Bounding-box rows:", len(boxes))
    print("Unique radiologists:", boxes["rad_id"].nunique())
    print("\nCoordinate ranges:")
    print(boxes[["x_min", "y_min", "x_max", "y_max"]].describe())
    print("\nAnnotations:")
    print(
        boxes[["rad_id", "class_name", "x_min", "y_min", "x_max", "y_max"]]
        .to_string(index=False)
    )


DICOM dimensions: 2880 x 2304
Bounding-box rows: 7
Unique radiologists: 3

Coordinate ranges:
             x_min        y_min        x_max        y_max
count     7.000000     7.000000     7.000000     7.000000
mean    996.911848  1196.834420  1681.834263  1591.501439
std     227.330087   415.693643   254.575268   485.201659
min     745.325012   731.721985  1467.819946  1014.059998
25%     759.618988   763.969513  1479.599976  1114.690002
50%    1153.599976  1497.079956  1484.729980  1830.680054
75%    1176.380005  1524.929993  1946.019958  2003.570007
max    1207.459961  1571.239990  1969.050049  2059.250000

Annotations:
rad_id         class_name       x_min       y_min       x_max       y_max
   R10       Cardiomegaly  761.935974 1519.319946 1933.449951 1977.530029
   R10 Aortic enlargement 1155.739990  758.953003 1467.819946 1014.059998
    R8       Cardiomegaly  745.325012 1571.239990 1969.050049 2059.250000
    R8 Aortic enlargement 1207.459961  768.986023 1484.729980 1175.660034


In [17]:
print("Current working directory:", Path.cwd())

for p in ROOT.rglob("image_labels_train.csv"):
    print("Found labels:", p)

for p in ROOT.rglob("annotations_train.csv"):
    print("Found annotations:", p)

Current working directory: c:\Users\johnm\chest-xray-ai-annotation\notebooks


In [19]:
from pathlib import Path

ROOT = Path.cwd().parent

LABEL_PATH = ROOT / "data" / "annotations" / "image_labels_train.csv"
ANNOTATION_PATH = ROOT / "data" / "annotations" / "annotations_train.csv"

PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Labels found:", LABEL_PATH.exists())
print("Annotations found:", ANNOTATION_PATH.exists())
print("Output folder:", PROCESSED_DIR)

Project root: c:\Users\johnm\chest-xray-ai-annotation
Labels found: True
Annotations found: True
Output folder: c:\Users\johnm\chest-xray-ai-annotation\data\processed


In [20]:

import pandas as pd
import numpy as np
from pathlib import Path

# ==================================================
# 1. PATHS
# ==================================================

ROOT = Path.cwd().parent

LABEL_PATH = ROOT / "data" / "annotations" / "image_labels_train.csv"
ANNOTATION_PATH = ROOT / "data" / "annotations" / "annotations_train.csv"
PROCESSED_DIR = ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# ==================================================
# 2. LOAD CSV FILES
# ==================================================

labels_df = pd.read_csv(LABEL_PATH)
annotations_df = pd.read_csv(ANNOTATION_PATH)

TARGET_CLASSES = [
    "Aortic enlargement",
    "Cardiomegaly",
    "Pleural thickening",
    "Pulmonary fibrosis",
    "Lung Opacity"
]

print("Labels:", labels_df.shape)
print("Annotations:", annotations_df.shape)

# ==================================================
# 3. AGGREGATE CLASSIFICATION LABELS
# ==================================================

# Positive if at least one radiologist marks the finding positive.
image_labels = (
    labels_df.groupby("image_id")[TARGET_CLASSES]
    .max()
    .fillna(0)
    .astype(int)
)

# ==================================================
# 4. MATCH POSITIVE LABELS WITH BOUNDING BOXES
# ==================================================

target_annotations = annotations_df[
    annotations_df["class_name"].isin(TARGET_CLASSES)
].copy()

bbox_pairs = set(
    zip(
        target_annotations["image_id"],
        target_annotations["class_name"]
    )
)

eligible_ids = []

for image_id, row in image_labels.iterrows():

    positive_classes = [
        cls for cls in TARGET_CLASSES if row[cls] == 1
    ]

    if positive_classes and all(
        (image_id, cls) in bbox_pairs
        for cls in positive_classes
    ):
        eligible_ids.append(image_id)

eligible_df = image_labels.loc[eligible_ids].copy()

print("\nEligible images:", len(eligible_df))

if len(eligible_df) < 300:
    raise ValueError("Fewer than 300 eligible images available.")

# ==================================================
# 5. MULTI-LABEL SAMPLING (REPRODUCIBLE)
# ==================================================

rng = np.random.default_rng(42)

selected_ids = []
selected_set = set()

# Aim for at least 60 images per finding.
for cls in TARGET_CLASSES:

    current_count = (
        int(eligible_df.loc[selected_ids, cls].sum())
        if selected_ids else 0
    )

    candidates = eligible_df.index[
        (eligible_df[cls] == 1) &
        (~eligible_df.index.isin(selected_set))
    ].to_numpy()

    rng.shuffle(candidates)

    for image_id in candidates:

        if current_count >= 60:
            break

        selected_ids.append(image_id)
        selected_set.add(image_id)
        current_count += 1

# Fill remaining slots to exactly 300 unique images.
remaining_ids = eligible_df.index[
    ~eligible_df.index.isin(selected_set)
].to_numpy()

rng.shuffle(remaining_ids)

selected_ids.extend(
    remaining_ids[:300 - len(selected_ids)].tolist()
)

if len(selected_ids) != 300 or len(set(selected_ids)) != 300:
    raise ValueError("Could not generate exactly 300 unique images.")

# ==================================================
# 6. CREATE MASTER IMAGE MANIFEST
# ==================================================

manifest = eligible_df.loc[selected_ids].reset_index()

manifest.insert(1, "sample_order", range(1, 301))

manifest["number_of_target_findings"] = (
    manifest[TARGET_CLASSES].sum(axis=1)
)

# Add bounding-box counts per class.
for cls in TARGET_CLASSES:

    counts = (
        target_annotations[
            target_annotations["class_name"] == cls
        ]
        .groupby("image_id")
        .size()
    )

    manifest[f"bbox_count_{cls}"] = (
        manifest["image_id"]
        .map(counts)
        .fillna(0)
        .astype(int)
    )

# ==================================================
# 7. EXTRACT MATCHING BOUNDING BOXES
# ==================================================

selected_annotations = target_annotations[
    target_annotations["image_id"].isin(selected_ids)
].copy()

selected_annotations = selected_annotations.sort_values(
    ["image_id", "class_name", "rad_id"]
).reset_index(drop=True)

# ==================================================
# 8. CLASS SUMMARY
# ==================================================

summary_rows = []

for cls in TARGET_CLASSES:

    summary_rows.append({
        "finding": cls,
        "selected_positive_images": int(manifest[cls].sum()),
        "images_with_bounding_boxes": int(
            (manifest[f"bbox_count_{cls}"] > 0).sum()
        ),
        "total_bounding_box_annotations": int(
            (selected_annotations["class_name"] == cls).sum()
        )
    })

class_summary = pd.DataFrame(summary_rows)

# ==================================================
# 9. SAVE FILES TO PROCESSED FOLDER
# ==================================================

manifest.to_csv(
    PROCESSED_DIR / "vindr_300_image_manifest.csv",
    index=False
)

selected_annotations.to_csv(
    PROCESSED_DIR / "vindr_300_bounding_boxes.csv",
    index=False
)

class_summary.to_csv(
    PROCESSED_DIR / "vindr_300_class_summary.csv",
    index=False
)

# ==================================================
# 10. FINAL VALIDATION AND REPORT
# ==================================================

assert manifest["image_id"].nunique() == 300
assert selected_annotations["image_id"].nunique() == 300

print("\n========== DATASET SELECTION COMPLETE ==========")
print("Project root:", ROOT)
print("Unique images selected:", manifest["image_id"].nunique())
print("Bounding-box records:", len(selected_annotations))

print("\nCLASS SUMMARY:")
display(class_summary)

print("\nFILES SAVED:")
for filename in [
    "vindr_300_image_manifest.csv",
    "vindr_300_bounding_boxes.csv",
    "vindr_300_class_summary.csv"
]:
    print(PROCESSED_DIR / filename)

print("\nSUCCESS: All three files saved!")


Labels: (45000, 30)
Annotations: (69052, 7)

Eligible images: 4372

========== DATASET SELECTION COMPLETE ==========
Project root: c:\Users\johnm\chest-xray-ai-annotation
Unique images selected: 300
Bounding-box records: 1799

CLASS SUMMARY:


,finding,selected_positive_images,images_with_bounding_boxes,total_bounding_box_annotations
0,Aortic enlargement,218,218,522
1,Cardiomegaly,163,163,379
2,Pleural thickening,146,146,359
3,Pulmonary fibrosis,123,123,337
4,Lung Opacity,108,108,202



FILES SAVED:
c:\Users\johnm\chest-xray-ai-annotation\data\processed\vindr_300_image_manifest.csv
c:\Users\johnm\chest-xray-ai-annotation\data\processed\vindr_300_bounding_boxes.csv
c:\Users\johnm\chest-xray-ai-annotation\data\processed\vindr_300_class_summary.csv

SUCCESS: All three files saved!
